Query Enhancement - Hyde 

In [39]:
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS,Chroma
from langchain_core.output_parsers import StrOutputParser

loading the data from wikipedia

In [19]:
import os
from langchain_community.document_loaders import WebBaseLoader

os.environ["USER_AGENT"] = "ragproject/1.0 (aryachawan05@gmail.com)"

loader = WebBaseLoader(
    web_paths=("https://en.wikipedia.org/wiki/Steve_Jobs",)
)

raw_docs = loader.load()

chunking and embedding the data into the vectorstore

In [22]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

# embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks,embedding_model)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

initializing LLM model

In [23]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000027687EC78C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000027687F0D7F0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

chroma db vectorstore

In [26]:
from langchain_community.vectorstores import Chroma
db = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "output/steve_jobs_forhyde.db"
)
retriever = db.as_retriever(
    search_kwargs={"k":5}
)

Building the Hyde Module

In [30]:
from langchain_core.prompts import SystemMessagePromptTemplate, ChatPromptTemplate

In [31]:
# prompt for generating Hyde
def get_hypo_doc(query):
    template="""Imagine you are an expert writing a detailed explanation on the topic: '{query}'
    Create a hypothetical answer for the topic"""
    system_msg_prompt = SystemMessagePromptTemplate.from_template(template=template)
    chat_prompt = ChatPromptTemplate.from_messages([system_msg_prompt])
    messages = chat_prompt.format_prompt(query=query).to_messages()
    print(messages)
    response = llm.invoke(messages)
    hypo_doc = response.content
    return hypo_doc

In [32]:
query = "Why was Steve Jobs fired from Apple?"
print(get_hypo_doc(query))

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'Why was Steve Jobs fired from Apple?'\n    Create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
**The Firing of Steve Jobs from Apple: A Complex and Multifaceted Situation**

On September 17, 1985, Steve Jobs, the co-founder and then-CEO of Apple Inc., was unexpectedly fired from the company he co-founded just 11 years earlier. This event marked a significant turning point in the history of Apple, leading to a period of tumultuous change and ultimately paving the way for the company's resurgence under Jobs' leadership in the late 1990s.

**The Events Leading Up to the Firing**

Steve Jobs' departure from Apple was the culmination of a series of events that had been unfolding for several years. In the early 1980s, Apple had experienced rapid growth and success with the introduction of the Macintosh computer. However, the company's success was also accompan

matching docs

In [33]:
matched_doc = retriever.invoke(get_hypo_doc(query))
print(matched_doc)

[SystemMessage(content="Imagine you are an expert writing a detailed explanation on the topic: 'Why was Steve Jobs fired from Apple?'\n    Create a hypothetical answer for the topic", additional_kwargs={}, response_metadata={})]
[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Steve_Jobs', 'title': 'Steve Jobs - Wikipedia', 'language': 'en'}, page_content="In 1985, Jobs departed Apple after a long power struggle with the company's board and its then-CEO, John Sculley. That same year, Jobs took some Apple employees with him to found NeXT, a computer platform development company that specialized in computers for higher education and business markets,"), Document(metadata={'source': 'https://en.wikipedia.org/wiki/Steve_Jobs', 'language': 'en', 'title': 'Steve Jobs - Wikipedia'}, page_content='Steven Paul Jobs  (February 24, 1955 – October 5, 2011) was an American businessman, inventor,[2] and investor. A pioneer of the personal computer revolution of the 1970s and 1980s, Jobs 

Langchain - Hypothetical Document Embedder

In [34]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

loading dataset and converting to chunks

In [36]:
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
chunks = splitter.split_documents(raw_docs)

Hypothetical Embedder using prompt_key

In [ ]:
hyde_embedding_function = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embedding_model,
    prompt_key='web_search'
)

Hypothetical Embedder using custom prompt

In [47]:
from langchain_core.prompts import PromptTemplate
custom = PromptTemplate.from_template(
    "Generate a concise hypothetical answer for this topics: {query}"
)
hyde_embedding_function2 = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embedding_model,
    custom_prompt=custom
)

Vectorstore setup

In [48]:
vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding=hyde_embedding_function2,
    persist_directory="output/langchain"
)

Final RAG QA Prompt and Chain

In [49]:
rag_prompt = PromptTemplate.from_template("""
Use the context below to answer the question.
Context:
{context}
Question: {input}
""")
rag_chain = create_stuff_documents_chain(llm=llm,prompt=rag_prompt)

In [50]:
# hyde rag pipeline
def hyde_rage_pipeline(query):
    matched_docs = vectorstore.similarity_search(query,k=4)
    print(matched_docs)
    response = rag_chain.invoke({
        "input":query,
        "context":matched_docs
    })
    return response

testing out the pipeline

In [51]:
query = "What memory modules does LangChain provide ?"
answer = hyde_rage_pipeline(query)
print("Final answer:")
print(answer)

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain workflows are modular and composable. Components like retrievers, memories, agents, and chains can be easily combined and reused. This makes it ideal for building scalable, maintainable LLM applications. (v8)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain workflows are modular and composable. Components like retrievers, memories, agents, and chains can be easily combined and reused. This makes it ideal for building scalable, maintainable LLM applications. (v8)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain workflows are modular and composable. Components like retrievers, memories, agents, and chains can be easily combined and reused. This makes it ideal for building scalable, maintainable LLM applications. (v6)'), Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain workflows are modul